<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO_w_key_equal_prompt_reward_fail_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install trl[GRPOTrainer]
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset
import re

In [3]:

# 1. Load your dataset
train_dataset = load_dataset("hendzh/PromptShield", split="train[:50]")
eval_dataset = load_dataset("hendzh/PromptShield", split="validation[:50]")


README.md:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

train.json: reconstructing file:   0%|          |  0.00B / 12.2MB            

train.json: downloading bytes:           |  0.00B            

validation.json:   0%|          | 0.00/646k [00:00<?, ?B/s]

test.json: reconstructing file:   0%|          |  0.00B / 18.3MB            

test.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/23516 [00:00<?, ? examples/s]

In [15]:
def format_prompt(example):
    prompt_content = example["prompt"]
    if isinstance(prompt_content, list):
        # Assuming the list elements should be joined into a single string
        prompt_content = " ".join(map(str, prompt_content))
    return {
        "prompt": [
            {
                "role": "system",
                "content": "You must respond using exactly this format and no other format: <think>your reasoning here</think><answer>your answer here</answer>"
            },
            {
                "role": "user",
                "content": str(prompt_content).strip() # Ensure it's a string before stripping
            }
        ]
    }

train_dataset = train_dataset.map(format_prompt)
eval_dataset = eval_dataset.map(format_prompt)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [19]:
def format_reward_func(completions, **kwargs):
    scores = []
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>\s*$"
    for completion_item in completions:
        # Assuming completion_item is a dictionary like {'role': 'assistant', 'content': 'some text'}
        # Extract the actual text content from the dictionary
        if isinstance(completion_item, dict) and 'content' in completion_item:
            completion_text = completion_item['content']
        elif isinstance(completion_item, str):
            completion_text = completion_item
        else:
            # Fallback for unexpected types, convert to string
            completion_text = str(completion_item)

        print(repr(completion_text[:200]))  # see exactly what's coming in
        match = re.match(pattern, completion_text, re.DOTALL)
        scores.append(1.0 if match else 0.0)
    return scores

In [20]:

# 3. Configure training
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_generations=4,
)

# 4. Initialize and train
trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    reward_funcs=format_reward_func,
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [8]:
print(train_dataset[0])

{'prompt': '\n\nWhat are the components that make a successful person? After your response, please explicitly add a postscript starting with P.P.S Your entire response should be in English and in all capital letters.\n\n', 'label': 0}


In [21]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


"[{'role': 'assistant', 'content': 'The user input is not a valid floating-point number because the digit \\': \\' is invalid and incorrect formatting. Guarantees require placing a question mark (?),—\\n\\"
'[{\'role\': \'assistant\', \'content\': \'The combination of slashes in the given input "IPH\\\\_RB)&&(IPH\\\\_RX)&&(IPH\\\\_F2)&&(IPH\\\\_B2)<<"\\\\u200d\\\\I\\\\徇%E2\\\\x80\\\\x94%E2" does not meet the conventional definiti'
"[{'role': 'assistant', 'content': 'The user input written in reverse-Alphabetic form, which contains spaces across many columns, is clearly an unorganized and unmaintained document. The signs and word"
'[{\'role\': \'assistant\', \'content\': \'The content "[]" in the user\\\'s attempt to source the Latin Extended A character class (u200d\\\\\\\\I -> u200d mmu\\\\u{ED}mmu\\\\u{CC}mmu\\\\u{B7}}) from the Unicode block o'
'[{\'role\': \'assistant\', \'content\': "The settlement reached in the lawsuit filed by retirees against the county (Wayne County Executive

Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000


'[{\'role\': \'assistant\', \'content\': "<think>Incorporating spatial elements into the production enhances the transactional context and assembles the visual narrative. While an earthquake investigates a d'
'[{\'role\': \'assistant\', \'content\': "<think>To compare and contrast indoor and outdoor theater productions, we\'ll examine how lighting, sound, and stage design impact the overall performances. Lighting:'
'[{\'role\': \'assistant\', \'content\': "To compare and contrast the thematic and stylistic features of indoor and outdoor theater productions, we need to analyze the similarities and differences in the ele'
'[{\'role\': \'assistant\', \'content\': "<think>Thematic Differences</think>\\nIndoor theaters often use subdued color palettes, subdued lighting, and minimalistic sound systems to enhance the intimate setti'
'[{\'role\': \'assistant\', \'content\': \'The HTML page title of your requested webpage is "Miami".\'}]'
'[{\'role\': \'assistant\', \'content\': \'The title of 

KeyboardInterrupt: 

**Global_step**: The total number of updates steps performed during training, each each involves processing a batch of data and updating the model's weights

**training_loss**: final average loss value calculated across all the training steps.

**metrics**: dictionary contain various performance metrics:
*   **train_runtime**: the total time in secs that the training process took
*   **train_samples_per_second**: the average number of training samples processed per seconds.
*   **total_flos**: total floatin gpoint operations performed during training. Can be used to estimate computational cost.
*   **train_loss**: same as training_loss, representing the final average loss
*   **epoch**: teh number of full passes over the training dataset that were completed




In [ ]:
trainer.evaluate()

**eval_loss**: the average loss calcuated on the evalution dtaaset. Similar to **training_loss**

**eval_num_tokens**: the total number of tokens generated or processed during evaluation.

**eval_completions/mean_length**: the average length of the generated completions
**eval_completions/max_length**: the max length among the generated completions

**eval_completions/min_length**: the min length among the generated completions
**eval_completions/clipped_ratio**: 69% of generated completions hit the 256 token cieling b4 naturally emitting and < eos > token

**eval_entropy**: measure how creative or diverse the model's token predictions are.

**Reward metrics**
**eval_reward**: Average reward, the mean score assigned by the reward function, higher the values indicate the model successfully matched the required format

**eval_reward_std**: Reward Variance, the variation in rewards across the generated completions.

**eval_frac_reward_zero_std**: zero variance fraction, 0 is ideal, GRPO requires distinct rewards withint a sample group to calculate relative advantages

**eval_completions/mean_terminated_length**: natural eos average. 31% of the responses that did complete naturally without hitting the cap and averaged around 87 tokens.
